In [ ]:
from hcc_release_paths import project_path as _hcc_path
from pathlib import Path
import pandas as pd
import json
from datetime import datetime

ROOT = Path(_hcc_path('LIHC_RNA_Step17_Bulk_Freeze'))

NOTEBOOK_DIR = ROOT / "Notebook"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"

for p in [NOTEBOOK_DIR, RESULTS_DIR, FIGURES_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Results:", RESULTS_DIR)
print("Bulk freeze notebook initialized.")

In [ ]:
OriginalTop10 = [
    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "AC015871.6",
    "LINC00324",
    "ACOXL",
    "PTMAP4",
    "ANKRD20A7P",
]

Dynamic9 = [
    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "LINC00324",
    "PTMAP4",
    "ANKRD20A7P",
    "BOP1",
]

StableGenes10 = [
    "HOXD9",
    "RPSAP53",
    "RFESD",
    "LINC00324",
    "TRGC1",
    "ANKRD20A7P",
    "FAM227B",
    "BOP1",
    "GLYATL1",
    "PRR11",
]

StableBio7 = [
    "mTORC1 Signaling",
    "Checkpoint_TNFRSF4",
    "Cell_B_cells",
    "Function_HLA_activity",
    "Checkpoint_CD276",
    "Pancreas Beta Cells",
    "Spermatogenesis",
]

Hub18 = {
    "Blue": [
        "CENPA", "KIF2C", "CCNB1",
        "CDC20", "PRR11", "CDCA8"
    ],
    "Red": [
        "SLC52A2", "RPL8", "BOP1",
        "ZFAS1", "SNHG6", "RPL7P9"
    ],
    "Brown": [
        "CYP8B1", "GLYATL1", "PLG",
        "ACSM2A", "ABAT", "MAT1A"
    ],
}

print("OriginalTop10:", len(OriginalTop10))
print("Dynamic9:", len(Dynamic9))
print("StableGenes10:", len(StableGenes10))
print("StableBio7:", len(StableBio7))
print("Hub18:", sum(len(v) for v in Hub18.values()))

In [ ]:
def save_list(items, filename):
    path = RESULTS_DIR / filename
    path.write_text("\n".join(items) + "\n", encoding="utf-8")
    return path

save_list(OriginalTop10, "OriginalTop10_locked.txt")
save_list(Dynamic9, "Dynamic9_locked.txt")
save_list(StableGenes10, "StableGenes10_locked.txt")
save_list(StableBio7, "StableBio7_locked.txt")

hub_lines = []
for module, genes in Hub18.items():
    for gene in genes:
        hub_lines.append(f"{module}\t{gene}")

(RESULTS_DIR / "Hub18_locked.txt").write_text(
    "Module\tGene\n" + "\n".join(hub_lines) + "\n",
    encoding="utf-8"
)

print("Locked feature files saved.")

In [ ]:
model_summary = {
    "Primary_Model": "Original Top10 + Unpenalized CoxPH",
    "Sensitivity_Model": "Original Top10 + ElasticNet",
    "Alternative_Model": "Dynamic9 + Unpenalized CoxPH",
    "Biological_Integration_Model": "Genes10_Bio7 + ElasticNet",

    "Development_N": 238,
    "Development_Events": 82,

    "Development_RepeatedCV_Cindex": 0.7444,
    "Development_Apparent_Cindex": 0.7646,
    "Bootstrap_Mean_Optimism": 0.0268,
    "Optimism_Corrected_Cindex": 0.7378,

    "Holdout_N": 102,
    "Holdout_Events": 35,
    "Holdout_Cindex": 0.7998,
    "Holdout_Cindex_CI_Lower": 0.6969,
    "Holdout_Cindex_CI_Upper": 0.8778,

    "AUC_1y": 0.841,
    "AUC_2y": 0.876,
    "AUC_3y": 0.854,

    "Frozen_Development_Cutoff": 0.001977125611007005,

    "Holdout_KM_Low_N": 59,
    "Holdout_KM_Low_Events": 11,
    "Holdout_KM_High_N": 43,
    "Holdout_KM_High_Events": 24,

    "Holdout_HR": 4.51,
    "Holdout_HR_CI_Lower": 2.19,
    "Holdout_HR_CI_Upper": 9.26,
    "Holdout_Logrank_P": 7.919e-06,

    "Calibration_Slope": 0.899,
    "Calibration_Slope_CI_Lower": 0.571,
    "Calibration_Slope_CI_Upper": 1.227,

    "Brier_1y": 0.097,
    "Brier_2y": 0.137,
    "Brier_3y": 0.164,
}

model_df = pd.DataFrame(
    model_summary.items(),
    columns=["Metric", "Value"]
)

model_df.to_csv(
    RESULTS_DIR / "Bulk_Final_Model_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

model_df

In [ ]:
bulk_findings = [
    # subtype
    ["Subtype", "Metabolic", "229 patients", "Step13"],
    ["Subtype", "Proliferative", "136 patients", "Step13"],

    # high-risk pathways
    ["HighRisk_Pathway", "E2F Targets", "Higher in proliferative/high-risk", "Step14"],
    ["HighRisk_Pathway", "G2M Checkpoint", "Higher in proliferative/high-risk", "Step14"],
    ["HighRisk_Pathway", "MYC Targets", "Higher in proliferative/high-risk", "Step14"],
    ["HighRisk_Pathway", "Mitotic Spindle", "Higher in proliferative/high-risk", "Step14"],
    ["HighRisk_Pathway", "DNA Repair", "Higher in proliferative/high-risk", "Step14"],
    ["HighRisk_Pathway", "mTORC1 Signaling", "Higher in proliferative/high-risk", "Step14"],
    ["HighRisk_Pathway", "Unfolded Protein Response", "Higher in proliferative/high-risk", "Step14"],
    ["HighRisk_Pathway", "Wnt-related signaling", "Higher in proliferative/high-risk", "Step14"],
    ["HighRisk_Pathway", "PI3K/AKT/mTOR-related signaling", "Higher in proliferative/high-risk", "Step14"],

    # metabolic
    ["LowRisk_Pathway", "Peroxisome", "Higher in metabolic/low-risk", "Step14"],
    ["LowRisk_Pathway", "Adipogenesis", "Higher in metabolic/low-risk", "Step14"],
    ["LowRisk_Pathway", "Fatty Acid Metabolism", "Higher in metabolic/low-risk", "Step14"],
    ["LowRisk_Pathway", "Coagulation", "Higher in metabolic/low-risk", "Step14"],
    ["LowRisk_Pathway", "Bile Acid Metabolism", "Higher in metabolic/low-risk", "Step14"],

    # immune
    ["Immune", "TNFRSF4", "Higher in high-risk", "Step11"],
    ["Immune", "CD276", "Higher in high-risk", "Step11"],
    ["Immune", "VTCN1", "Higher in high-risk", "Step11"],
    ["Immune", "CD40LG", "Lower in high-risk", "Step11"],
    ["Immune", "BTLA", "Lower in high-risk", "Step11"],
    ["Immune", "CD160", "Lower in high-risk", "Step11"],
    ["Immune", "T-cell inflamed", "Lower in high-risk", "Step11"],
    ["Immune", "Cytolytic activity", "Lower in high-risk", "Step11"],
    ["Immune", "IFNG-related state", "Lower in high-risk", "Step11"],
]

bulk_findings_df = pd.DataFrame(
    bulk_findings,
    columns=["Category", "Feature", "Finding", "Source_Step"]
)

bulk_findings_df.to_csv(
    RESULTS_DIR / "Bulk_Biological_Findings.csv",
    index=False,
    encoding="utf-8-sig"
)

bulk_findings_df

In [ ]:
external_validation = pd.DataFrame([
    {
        "Dataset": "GSE14520",
        "Platform": "Microarray",
        "Top10_Coverage": "2/10",
        "Survival": "DFS/recurrence",
        "Role": "Biological validation only",
        "Status": "LOCKED"
    },
    {
        "Dataset": "GSE76427",
        "Platform": "Microarray",
        "Top10_Coverage": "3/10",
        "Survival": "OS/RFS",
        "Role": "Biological validation only",
        "Status": "LOCKED"
    },
    {
        "Dataset": "GSE138485",
        "Platform": "RNA-seq",
        "Top10_Coverage": "9/10",
        "Survival": "No reliable survival",
        "Role": "Paired tumor-normal molecular reproducibility",
        "Status": "LOCKED"
    },
    {
        "Dataset": "CHCC-HBV / HCCDB25",
        "Platform": "RNA-seq",
        "Top10_Coverage": "9/10",
        "Survival": "OS/RFS complete",
        "Role": "Biological/prognostic validation",
        "Status": "LOCKED"
    },
    {
        "Dataset": "GSE141198",
        "Platform": "Paired-end bulk RNA-seq",
        "Top10_Coverage": "Pending re-quantification",
        "Survival": "148/148 OS metadata; event field 94/54",
        "Role": "Raw FASTQ external-validation pilot",
        "Status": "ONGOING"
    }
])

external_validation.to_csv(
    RESULTS_DIR / "External_Validation_Status.csv",
    index=False,
    encoding="utf-8-sig"
)

external_validation

In [ ]:
anchors = []

# Original Top10
for g in OriginalTop10:
    anchors.append([
        "OriginalTop10",
        g,
        "Risk model feature",
        "Step8/15/16",
        "Patient-level prognostic anchor",
        "malignant_state_scoring"
    ])

# Hub18
for module, genes in Hub18.items():
    for g in genes:
        direction = {
            "Blue": "Higher in proliferative subtype",
            "Red": "Higher in proliferative subtype",
            "Brown": "Higher in metabolic subtype"
        }[module]

        anchors.append([
            f"WGCNA_{module}",
            g,
            direction,
            "Step12/13",
            "Bulk co-expression module hub",
            "malignant_state_scoring;GRN_candidate"
        ])

# pathways
high_pathways = [
    "E2F Targets",
    "G2M Checkpoint",
    "MYC Targets",
    "Mitotic Spindle",
    "DNA Repair",
    "mTORC1 Signaling",
    "Unfolded Protein Response",
    "Wnt-related signaling",
    "PI3K/AKT/mTOR-related signaling",
]

low_pathways = [
    "Peroxisome",
    "Adipogenesis",
    "Fatty Acid Metabolism",
    "Coagulation",
    "Bile Acid Metabolism",
]

for x in high_pathways:
    anchors.append([
        "HighRisk_Pathway",
        x,
        "Higher in high-risk/proliferative",
        "Step14",
        "High-risk biological program",
        "malignant_state_scoring;trajectory_anchor"
    ])

for x in low_pathways:
    anchors.append([
        "LowRisk_Pathway",
        x,
        "Higher in low-risk/metabolic",
        "Step14",
        "Metabolic biological program",
        "malignant_state_scoring;trajectory_anchor"
    ])

# immune
immune_features = {
    "TNFRSF4": "Higher in high-risk",
    "CD276": "Higher in high-risk",
    "VTCN1": "Higher in high-risk",
    "CD40LG": "Lower in high-risk",
    "BTLA": "Lower in high-risk",
    "CD160": "Lower in high-risk",
    "T-cell inflamed": "Lower in high-risk",
    "Cytolytic activity": "Lower in high-risk",
    "IFNG-related state": "Lower in high-risk",
}

for x, direction in immune_features.items():
    anchors.append([
        "Immune",
        x,
        direction,
        "Step11",
        "Risk-associated immune phenotype",
        "cell_cell_communication;spatial_validation"
    ])

anchor_df = pd.DataFrame(
    anchors,
    columns=[
        "Category",
        "Feature",
        "Direction",
        "Source_Step",
        "Interpretation",
        "Use_in_scRNA"
    ]
)

anchor_df.to_csv(
    RESULTS_DIR / "SingleCell_Anchor_Features.tsv",
    sep="\t",
    index=False
)

print(anchor_df.shape)
anchor_df.head(20)

In [ ]:
manifest = {
    "Freeze_Date": datetime.now().strftime("%Y-%m-%d"),
    "Project": "TCGA-LIHC Bulk RNA-seq",
    "Status": "FROZEN",

    "Primary_Model": "Original Top10 + Unpenalized CoxPH",

    "OriginalTop10": OriginalTop10,
    "Dynamic9": Dynamic9,
    "StableGenes10": StableGenes10,
    "StableBio7": StableBio7,
    "Hub18": Hub18,

    "Frozen_Cutoff": 0.001977125611007005,

    "Development": {
        "N": 238,
        "Events": 82,
        "RepeatedCV_Cindex": 0.7444,
        "OptimismCorrected_Cindex": 0.7378
    },

    "Holdout": {
        "N": 102,
        "Events": 35,
        "Cindex": 0.7998,
        "CI95": [0.6969, 0.8778]
    },

    "Subtype": {
        "Metabolic_N": 229,
        "Proliferative_N": 136
    },

    "External_Validation_Status": {
        "Strict_external_validation": "ONGOING",
        "Current_candidate": "GSE141198",
        "Pilot_status": "Reference download / requantification pending"
    },

    "Downstream_Main_Stage": "Single-cell RNA-seq",
    "Bulk_Results_Modification_After_Freeze": "Not allowed without explicit versioned amendment"
}

with open(
    RESULTS_DIR / "Bulk_Freeze_Manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, ensure_ascii=False, indent=4)

print("Manifest saved.")

In [ ]:
summary_rows = [
    ["Bulk_Status", "FROZEN"],
    ["Primary_Model", "Original Top10 + Unpenalized CoxPH"],
    ["OriginalTop10_N", len(OriginalTop10)],
    ["Development_N", 238],
    ["Development_Events", 82],
    ["Development_CV_Cindex", 0.7444],
    ["Optimism_Corrected_Cindex", 0.7378],
    ["Holdout_N", 102],
    ["Holdout_Events", 35],
    ["Holdout_Cindex", 0.7998],
    ["Frozen_Cutoff", 0.001977125611007005],
    ["Metabolic_N", 229],
    ["Proliferative_N", 136],
    ["Strict_External_Validation", "ONGOING"],
    ["Next_Main_Stage", "Single-cell RNA-seq"],
]

summary_df = pd.DataFrame(
    summary_rows,
    columns=["Item", "Value"]
)

summary_df.to_csv(
    RESULTS_DIR / "Bulk_Freeze_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_df

In [ ]:
print("=" * 70)
print("TCGA-LIHC BULK ANALYSIS FREEZE")
print("=" * 70)

print("\nBULK ANALYSIS STATUS = FROZEN")

print("\nPRIMARY MODEL:")
print("Original Top10 + Unpenalized CoxPH")

print("\nORIGINAL TOP10:")
for i, g in enumerate(OriginalTop10, 1):
    print(f"{i:02d}. {g}")

print("\nFROZEN DEVELOPMENT CUTOFF:")
print(0.001977125611007005)

print("\nSTRICT EXTERNAL VALIDATION:")
print("ONGOING — GSE141198 raw FASTQ re-quantification pilot")

print("\nNEXT MAIN STAGE:")
print("Single-cell RNA-seq analysis")

print("\nSingle-cell anchor table:")
print(RESULTS_DIR / "SingleCell_Anchor_Features.tsv")

print("\n" + "=" * 70)